## 1. Data Preprocessing
In this section, we perform feature selection and dimensionality reduction on the medulloblastoma dataset to improve machine learning model performance. The dataset consists of gene expression data where each sample represents a patient, and each feature corresponds to a gene.

The preprocessing techniques applied include:

+ **Variance Thresholding**
+ **LASSO Regression-based Feature Selection**
+ **Tree-Based Feature Selection (Random Forest)**
+ **Principal Component Analysis (PCA)**

## 1.1 Loading the Dataset
We start by loading the dataset, extracting gene features and sample labels, and setting up the preprocessing pipeline.

In [22]:
import pandas as pd
import numpy as np
import os

# Load the dataset
file_path = "data/medulloblastoma.tsv"
df = pd.read_csv(file_path, sep="\t")

# Extract features and labels
X = df.iloc[:, 1:].values.T  # Transpose so (samples, genes)
genes = df.iloc[:, 0]  # Gene names (IDs)
samples = df.columns[1:]  # Sample names

# Print dataset dimensions
print(f"Raw data: {X.shape[0]} samples, {X.shape[1]} features (genes)")

# Create output directory
output_dir = "data/preprocessed"
os.makedirs(output_dir, exist_ok=True)

Raw data: 73 samples, 54675 features (genes)


## 1.2 Variance Thresholding
**Goal**: Remove features (genes) that have low variance across samples, as they are unlikely to contribute to classification.

+ We apply different thresholds.
+ A higher threshold removes more genes.
+ Helps eliminate redundant or non-informative features.

In [23]:
from sklearn.feature_selection import VarianceThreshold

def variance_thresholding(X, genes, threshold):
    selector = VarianceThreshold(threshold=threshold)
    X_selected = selector.fit_transform(X)
    selected_genes = genes[selector.get_support()]

    num_features_left = X_selected.shape[1]
    df_selected = pd.DataFrame(X_selected, columns=selected_genes, index=samples)
    file_name = f"{output_dir}/fs_var_{threshold}_{num_features_left}.csv"
    df_selected.to_csv(file_name)

    print(f"[VarianceThreshold] threshold={threshold}, Features left: {num_features_left}, Saved: {file_name}")

thresholds = [0.03, 0.1, 0.3, 1.0, 3.0]
for threshold in thresholds:
    variance_thresholding(X, genes, threshold=threshold)

[VarianceThreshold] threshold=0.1, Features left: 50160, Saved: data/preprocessed/fs_var_0.1_50160.csv


## 1.3. LASSO Regression-Based Feature Selection
**Goal**: Select the most important genes using L1 regularization (LASSO).

+ LASSO shrinks the coefficients of less important genes to zero, leaving only the most relevant ones.
+ Hyperparameter `alpha` controls feature selection:
    + Lower `alpha` → keeps more genes.
    + Higher `alpha` → removes more genes.

In [13]:
from sklearn.linear_model import Lasso
from sklearn.feature_selection import SelectFromModel

def lasso_feature_selection(X, genes, alpha):
    lasso = Lasso(alpha=alpha, max_iter=10000).fit(X, np.random.rand(X.shape[0]))  # Dummy target
    selector = SelectFromModel(lasso, prefit=True)
    X_selected = selector.transform(X)

    selected_genes = genes[selector.get_support()]
    num_features_left = X_selected.shape[1]
    df_selected = pd.DataFrame(X_selected, columns=selected_genes, index=samples)
    file_name = f"{output_dir}/fs_reg_{alpha}_{num_features_left}.csv"
    df_selected.to_csv(file_name)

    print(f"[LASSO] alpha={alpha}, Features left: {num_features_left}, Saved: {file_name}")

alphas = [1e-2,1e-3,1e-4,1e-5,1e-6]
for alpha in alphas:
    lasso_feature_selection(X, genes, alpha=alpha)

[LASSO] alpha=0.0001, Features left: 94, Saved: data/preprocessed/fs_reg_0.0001_94.csv
[LASSO] alpha=0.0005, Features left: 80, Saved: data/preprocessed/fs_reg_0.0005_80.csv
[LASSO] alpha=0.001, Features left: 79, Saved: data/preprocessed/fs_reg_0.001_79.csv
[LASSO] alpha=0.01, Features left: 65, Saved: data/preprocessed/fs_reg_0.01_65.csv


## 1.4 Tree-Based Feature Selection (Random Forest)
**Goal**: Use Random Forest to identify the most important genes.

+ Decision trees rank features based on how useful they are in classification.
+ We apply different `n_estimators` (number of trees in the forest).

In [14]:
from sklearn.ensemble import RandomForestClassifier

def tree_based_feature_selection(X, genes, n_estimators):
    model = RandomForestClassifier(n_estimators=n_estimators, random_state=42)
    model.fit(X, np.random.randint(0, 4, X.shape[0]))  # Dummy target for feature importance
    selector = SelectFromModel(model, prefit=True)
    X_selected = selector.transform(X)

    selected_genes = genes[selector.get_support()]
    num_features_left = X_selected.shape[1]
    df_selected = pd.DataFrame(X_selected, columns=selected_genes, index=samples)
    file_name = f"{output_dir}/fs_tree_based_{n_estimators}_{num_features_left}.csv"
    df_selected.to_csv(file_name)

    print(f"[RandomForest] n_estimators={n_estimators}, Features left: {num_features_left}, Saved: {file_name}")

n_estimators_list = [25, 50, 100, 200, 300]
for n_estimator in n_estimators_list:
    tree_based_feature_selection(X, genes, n_estimators=n_estimator)

[RandomForest] n_estimators=50, Features left: 568, Saved: data/preprocessed/fs_tree_based_50_568.csv
[RandomForest] n_estimators=100, Features left: 1198, Saved: data/preprocessed/fs_tree_based_100_1198.csv
[RandomForest] n_estimators=200, Features left: 2254, Saved: data/preprocessed/fs_tree_based_200_2254.csv
[RandomForest] n_estimators=300, Features left: 3370, Saved: data/preprocessed/fs_tree_based_300_3370.csv


## 1.5 Principal Component Analysis (PCA)
**Goal**: Reduce feature dimensionality while retaining most of the variance.

+ PCA transforms the gene expression data into fewer components while preserving structure.
+ We apply different `n_components` values (73, 50, 30, 20).
    + `n_components` cannot be larger than `n_samples`

In [24]:
from sklearn.decomposition import PCA

def pca_dimensionality_reduction(X, n_components):
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X)

    df_pca = pd.DataFrame(X_pca, columns=[f"PCA_{i + 1}" for i in range(n_components)], index=samples)
    file_name = f"{output_dir}/pca_{n_components}_{n_components}.csv"
    df_pca.to_csv(file_name)

    print(f"[PCA] n_components={n_components}, Features left: {n_components}, Saved: {file_name}")

n_components_list = [1, 5, 10, 20, 30, 50, 73]
for n_components in n_components_list:
    pca_dimensionality_reduction(X, n_components=n_components)

[PCA] n_components=1, Features left: 1, Saved: data/preprocessed/pca_1_1.csv


# 2 Training Machine Learning Models
Now that we have preprocessed the gene expression data, we train four different machine learning models to classify medulloblastoma subtypes (G3, G4, SHH, WNT).

## Models Used
We will train the following models:

+ **Support Vector Machine (SVM)** → Good for high-dimensional data.
+ **Random Forest (RF)** → Provides feature importance rankings.
+ **XGBoost (XGB)** → Gradient boosting for robust performance.
+ **Logistic Regression (LR)** → A strong baseline model.

## 2.1 Loading Preprocessed Datasets
Each dataset has been preprocessed using different feature selection techniques, and we will train models on multiple feature sets.

In [28]:
import os
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Define the directory containing preprocessed datasets
data_dir = "data/preprocessed"

# Use all files or select specific ones
USE_ALL_FILES = True
if USE_ALL_FILES:
    files_to_use = [f for f in os.listdir(data_dir) if f.endswith(".csv")]
else:
    files_to_use = [
        "fs_tree_based_50_589.csv",
    ]


## 2.2 Extracting Labels from Sample Names
Since the labels are embedded in sample names (e.g., `G3_1` → `G3`), we extract them programmatically.

In [17]:
def extract_label(sample_name):
    match = re.match(r"([A-Za-z]+[0-9]*)", sample_name)
    return match.group(1) if match else "Unknown"

## 2.3 Defining Machine Learning Models
We use the following models with default parameters:

+ **SVM** (`linear kernel`)
+ **Random Forest** (`n_estimators=100`)
+ **XGBoost** (`eval_metric="mlogloss"`)
+ **Logistic Regression** (`max_iter=500`)

In [18]:
models = {
    "SVM": SVC(kernel="linear", class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(eval_metric="mlogloss"),
    "LogisticRegression": LogisticRegression(max_iter=500, class_weight="balanced"),
}

## 2.4 Training and Evaluating Models

For each dataset, we:

1. Load the dataset and extract features (`X`) and labels (`y`).
2. Encode the labels (`G3`, `G4`, `SHH`, `WNT`) numerically.
3. Split the data (80% training, 20% test).
4. Train each model and evaluate performance using:
    + Accuracy
    + F1-score for each class

In [29]:
# Store results
results = []

# Loop through each dataset and train models
for file in files_to_use:
    file_path = os.path.join(data_dir, file)

    if not os.path.exists(file_path):
        print(f"Skipping {file_path} (File Not Found)")
        continue

    # Load dataset
    df = pd.read_csv(file_path, index_col=0)  # Sample names are the index

    # Extract labels
    labels = df.index.to_series().apply(extract_label)

    # Encode labels
    unique_labels = sorted(labels.unique())  # ['G3', 'G4', 'SHH', 'WNT']
    label_map = {label: i for i, label in enumerate(unique_labels)}
    y = labels.map(label_map).values  # Convert labels to numeric values

    # Convert features
    X = df.values

    # Split into training and test sets (80/20 split)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # Train and evaluate each model
    for model_name, model in models.items():
        print(f"Training {model_name} on {file}...")

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        report = classification_report(y_test, y_pred, target_names=unique_labels, output_dict=True)

        results.append({
            "Dataset": file,
            "Model": model_name,
            "Accuracy": acc,
            "F1_G3": report["G3"]["f1-score"] if "G3" in report else 0,
            "F1_G4": report["G4"]["f1-score"] if "G4" in report else 0,
            "F1_SHH": report["SHH"]["f1-score"] if "SHH" in report else 0,
            "F1_WNT": report["WNT"]["f1-score"] if "WNT" in report else 0,
        })

        print(f"{model_name} on {file}: Accuracy = {acc:.4f}")

Training SVM on fs_reg_0.0001_85.csv...
SVM on fs_reg_0.0001_85.csv: Accuracy = 0.9333
Training RandomForest on fs_reg_0.0001_85.csv...
RandomForest on fs_reg_0.0001_85.csv: Accuracy = 0.9333
Training XGBoost on fs_reg_0.0001_85.csv...
XGBoost on fs_reg_0.0001_85.csv: Accuracy = 0.9333
Training LogisticRegression on fs_reg_0.0001_85.csv...
LogisticRegression on fs_reg_0.0001_85.csv: Accuracy = 0.9333
Training SVM on fs_reg_0.001_75.csv...
SVM on fs_reg_0.001_75.csv: Accuracy = 0.9333
Training RandomForest on fs_reg_0.001_75.csv...
RandomForest on fs_reg_0.001_75.csv: Accuracy = 0.8667
Training XGBoost on fs_reg_0.001_75.csv...
XGBoost on fs_reg_0.001_75.csv: Accuracy = 0.7333
Training LogisticRegression on fs_reg_0.001_75.csv...
LogisticRegression on fs_reg_0.001_75.csv: Accuracy = 0.9333
Training SVM on fs_reg_0.01_66.csv...
SVM on fs_reg_0.01_66.csv: Accuracy = 0.9333
Training RandomForest on fs_reg_0.01_66.csv...


E:\ProjectSoftwares\Anaconda\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
E:\ProjectSoftwares\Anaconda\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
E:\ProjectSoftwares\Anaconda\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


RandomForest on fs_reg_0.01_66.csv: Accuracy = 0.9333
Training XGBoost on fs_reg_0.01_66.csv...
XGBoost on fs_reg_0.01_66.csv: Accuracy = 0.8667
Training LogisticRegression on fs_reg_0.01_66.csv...
LogisticRegression on fs_reg_0.01_66.csv: Accuracy = 0.9333
Training SVM on fs_reg_1e-05_204.csv...
SVM on fs_reg_1e-05_204.csv: Accuracy = 1.0000
Training RandomForest on fs_reg_1e-05_204.csv...


E:\ProjectSoftwares\Anaconda\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
E:\ProjectSoftwares\Anaconda\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
E:\ProjectSoftwares\Anaconda\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


RandomForest on fs_reg_1e-05_204.csv: Accuracy = 0.9333
Training XGBoost on fs_reg_1e-05_204.csv...
XGBoost on fs_reg_1e-05_204.csv: Accuracy = 0.8667
Training LogisticRegression on fs_reg_1e-05_204.csv...
LogisticRegression on fs_reg_1e-05_204.csv: Accuracy = 1.0000
Training SVM on fs_reg_1e-06_649.csv...
SVM on fs_reg_1e-06_649.csv: Accuracy = 0.9333
Training RandomForest on fs_reg_1e-06_649.csv...
RandomForest on fs_reg_1e-06_649.csv: Accuracy = 0.8667
Training XGBoost on fs_reg_1e-06_649.csv...
XGBoost on fs_reg_1e-06_649.csv: Accuracy = 0.8667
Training LogisticRegression on fs_reg_1e-06_649.csv...
LogisticRegression on fs_reg_1e-06_649.csv: Accuracy = 0.9333
Training SVM on fs_tree_based_100_1145.csv...
SVM on fs_tree_based_100_1145.csv: Accuracy = 0.9333
Training RandomForest on fs_tree_based_100_1145.csv...
RandomForest on fs_tree_based_100_1145.csv: Accuracy = 0.9333
Training XGBoost on fs_tree_based_100_1145.csv...
XGBoost on fs_tree_based_100_1145.csv: Accuracy = 0.8667
Train

...

XGBoost on fs_tree_based_50_559.csv: Accuracy = 0.8667
Training LogisticRegression on fs_tree_based_50_559.csv...
LogisticRegression on fs_tree_based_50_559.csv: Accuracy = 0.9333
Training SVM on fs_var_0.03_54531.csv...
SVM on fs_var_0.03_54531.csv: Accuracy = 0.9333
Training RandomForest on fs_var_0.03_54531.csv...
RandomForest on fs_var_0.03_54531.csv: Accuracy = 0.9333
Training XGBoost on fs_var_0.03_54531.csv...
XGBoost on fs_var_0.03_54531.csv: Accuracy = 0.8667
Training LogisticRegression on fs_var_0.03_54531.csv...
LogisticRegression on fs_var_0.03_54531.csv: Accuracy = 0.9333
Training SVM on fs_var_0.1_50160.csv...
SVM on fs_var_0.1_50160.csv: Accuracy = 0.9333
Training RandomForest on fs_var_0.1_50160.csv...
RandomForest on fs_var_0.1_50160.csv: Accuracy = 0.9333
Training XGBoost on fs_var_0.1_50160.csv...
XGBoost on fs_var_0.1_50160.csv: Accuracy = 0.8667
Training LogisticRegression on fs_var_0.1_50160.csv...
LogisticRegression on fs_var_0.1_50160.csv: Accuracy = 0.9333
Trai

E:\ProjectSoftwares\Anaconda\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
E:\ProjectSoftwares\Anaconda\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
E:\ProjectSoftwares\Anaconda\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


XGBoost on fs_var_3_75.csv: Accuracy = 0.8667
Training LogisticRegression on fs_var_3_75.csv...
LogisticRegression on fs_var_3_75.csv: Accuracy = 0.9333
Training SVM on pca_10_10.csv...
SVM on pca_10_10.csv: Accuracy = 0.9333
Training RandomForest on pca_10_10.csv...
RandomForest on pca_10_10.csv: Accuracy = 0.9333
Training XGBoost on pca_10_10.csv...
XGBoost on pca_10_10.csv: Accuracy = 0.9333
Training LogisticRegression on pca_10_10.csv...
LogisticRegression on pca_10_10.csv: Accuracy = 0.9333
Training SVM on pca_1_1.csv...
SVM on pca_1_1.csv: Accuracy = 0.8667
Training RandomForest on pca_1_1.csv...
RandomForest on pca_1_1.csv: Accuracy = 0.8000
Training XGBoost on pca_1_1.csv...


E:\ProjectSoftwares\Anaconda\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
E:\ProjectSoftwares\Anaconda\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
E:\ProjectSoftwares\Anaconda\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


XGBoost on pca_1_1.csv: Accuracy = 0.7333
Training LogisticRegression on pca_1_1.csv...
LogisticRegression on pca_1_1.csv: Accuracy = 0.8667
Training SVM on pca_20_20.csv...
SVM on pca_20_20.csv: Accuracy = 0.9333
Training RandomForest on pca_20_20.csv...
RandomForest on pca_20_20.csv: Accuracy = 0.8667
Training XGBoost on pca_20_20.csv...
XGBoost on pca_20_20.csv: Accuracy = 0.9333
Training LogisticRegression on pca_20_20.csv...
LogisticRegression on pca_20_20.csv: Accuracy = 0.9333
Training SVM on pca_30_30.csv...
SVM on pca_30_30.csv: Accuracy = 0.9333
Training RandomForest on pca_30_30.csv...
RandomForest on pca_30_30.csv: Accuracy = 0.9333
Training XGBoost on pca_30_30.csv...
XGBoost on pca_30_30.csv: Accuracy = 0.9333
Training LogisticRegression on pca_30_30.csv...
LogisticRegression on pca_30_30.csv: Accuracy = 0.9333
Training SVM on pca_50_50.csv...
SVM on pca_50_50.csv: Accuracy = 0.9333
Training RandomForest on pca_50_50.csv...
RandomForest on pca_50_50.csv: Accuracy = 0.8667

## 2.5 Saving results

The results are saved to `ml_results.csv`for further analysis.

In [30]:
results_df = pd.DataFrame(results)
results_df.to_csv("ml_results.csv", index=False)

print("✅ Model training and evaluation completed! Results saved to ml_results.csv")

✅ Model training and evaluation completed! Results saved to ml_results.csv


In [31]:
# Initialize model win counters
models = ["SVM", "RandomForest", "XGBoost", "LogisticRegression"]
best_model_counts = {model: 0 for model in models}  # Shared wins
exclusive_model_counts = {model: 0 for model in models}  # Exclusive wins

# Iterate through each dataset
for dataset, group in results_df.groupby("Dataset"):
    max_accuracy = group["Accuracy"].max()  # Get the highest accuracy for this dataset
    best_models = group[group["Accuracy"] == max_accuracy]["Model"].unique()  # Get all models with max accuracy

    # Count all models that achieved the highest accuracy (shared wins)
    for model in best_models:
        best_model_counts[model] += 1

    # If only one model has the highest accuracy, count as exclusive win
    if len(best_models) == 1:
        exclusive_model_counts[best_models[0]] += 1

# Print results for shared wins
print("\n🏆 Best Model Performance Count Across Datasets (Considering Ties) 🏆")
for model, count in best_model_counts.items():
    print(f"{model}: {count} times (Shared)")

# Print results for exclusive wins
print("\n🥇 Exclusive Best Model Count (No Ties) 🥇")
for model, count in exclusive_model_counts.items():
    print(f"{model}: {count} times (Exclusive)")


🏆 Best Model Performance Count Across Datasets (Considering Ties) 🏆
SVM: 20 times (Shared)
RandomForest: 13 times (Shared)
XGBoost: 8 times (Shared)
LogisticRegression: 21 times (Shared)

🥇 Exclusive Best Model Count (No Ties) 🥇
SVM: 0 times (Exclusive)
RandomForest: 1 times (Exclusive)
XGBoost: 0 times (Exclusive)
LogisticRegression: 0 times (Exclusive)
